# MLOps Assignment 2 — LLMOps

## API Cost, Latency, and Prompt Version Management

**Scenario:** EduAssist AI — Academic Question Answering Assistant

**LLM Provider:** Google Gemini API

**Experiment:** `eduassist_llmops_assignment`

### Objectives
1. Measure API latency for five academic questions.
2. Extract token usage and calculate API costs.
3. Register and evaluate three prompt versions in MLflow.
4. Validate JSON responses and promote the best prompt.

All three prompt versions will be evaluated using the same five questions.


In [1]:
from pathlib import Path
import os
import time
from datetime import datetime, timezone

import pandas as pd
import mlflow

from dotenv import load_dotenv
from google import genai
from google.genai import types
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

load_dotenv(PROJECT_ROOT / ".env")

API_KEY = os.getenv("GEMINI_API_KEY")
MODEL_NAME = os.getenv("GEMINI_MODEL")
TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI")

assert API_KEY, "GEMINI_API_KEY is missing"
assert MODEL_NAME, "GEMINI_MODEL is missing"
assert TRACKING_URI, "MLFLOW_TRACKING_URI is missing"

mlflow.set_tracking_uri(TRACKING_URI)

experiment = mlflow.get_experiment_by_name(
    "eduassist_llmops_assignment"
)

assert experiment is not None, "MLflow experiment not found"

print("Project root:", PROJECT_ROOT)
print("Gemini model:", MODEL_NAME)
print("API key: CONFIGURED")
print("MLflow tracking URI:", TRACKING_URI)
print("MLflow experiment ID:", experiment.experiment_id)
print("Environment verification: PASSED")


Project root: /Volumes/MLOpsWorkspace/eduassist-llmops-assignment2
Gemini model: gemini-3.5-flash-lite
API key: CONFIGURED
MLflow tracking URI: http://127.0.0.1:5050
MLflow experiment ID: 1
Environment verification: PASSED


## Task 1 — API Calls and Latency Measurement

We evaluate five undergraduate-level academic questions.

The same dataset will be reused for prompt versions v1, v2, and v3.

### Prompt v1 — Basic

Answer the following student question clearly.

Question: {{question}}

Latency will be measured using `time.perf_counter()`.
Request timestamps will be recorded in UTC.


In [2]:
questions = [
    {
        "question_id": "Q1",
        "question": "What is the difference between supervised and unsupervised learning?"
    },
    {
        "question_id": "Q2",
        "question": "Explain overfitting in machine learning and one way to reduce it."
    },
    {
        "question_id": "Q3",
        "question": "What is gradient descent, and why is it used to train neural networks?"
    },
    {
        "question_id": "Q4",
        "question": "What is the difference between precision and recall in classification?"
    },
    {
        "question_id": "Q5",
        "question": "What is retrieval-augmented generation (RAG), and why is it useful?"
    }
]

PROMPT_V1 = (
    "Answer the following student question clearly.\n"
    "Question: {{question}}"
)

assert len(questions) == 5
assert len({q["question_id"] for q in questions}) == 5

print("Total academic questions:", len(questions))
print("Selected prompt: v1 — Basic")

display(pd.DataFrame(questions))

print("\nExample rendered prompt:\n")
print(PROMPT_V1.replace("{{question}}", questions[0]["question"]))


Total academic questions: 5
Selected prompt: v1 — Basic


,question_id,question
0,Q1,What is the difference between supervised and ...
1,Q2,Explain overfitting in machine learning and on...
2,Q3,"What is gradient descent, and why is it used t..."
3,Q4,What is the difference between precision and r...
4,Q5,"What is retrieval-augmented generation (RAG), ..."



Example rendered prompt:

Answer the following student question clearly.
Question: What is the difference between supervised and unsupervised learning?


In [7]:

# Task 1: Call an LLM and Measure Latency


# Task 1 — Gemini API Calls and Latency Measurement

from pathlib import Path
from datetime import datetime, timezone
import time
import pandas as pd

OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_RESULTS_PATH = OUTPUT_DIR / "task1_v1_raw.csv"

MAX_OUTPUT_TOKENS = 2048

RESULT_COLUMNS = [
    "question_id",
    "question",
    "prompt_version",
    "model_name",
    "start_time_utc",
    "end_time_utc",
    "latency_seconds",
    "response_status",
    "finish_reason",
    "input_tokens",
    "output_tokens",
    "thinking_tokens",
    "total_tokens",
    "response_text",
    "error_type",
    "max_output_tokens"
]

# Load previously saved measurements.
# Completed requests will not be repeated.
saved_results = {}

if RAW_RESULTS_PATH.exists():
    existing = pd.read_csv(
        RAW_RESULTS_PATH,
        keep_default_na=False
    )

    saved_results = {
        str(row["question_id"]): row
        for row in existing.to_dict("records")
    }


def is_complete(record):
    return (
        record.get("response_status") == "success"
        and record.get("finish_reason") == "STOP"
        and bool(str(record.get("response_text", "")).strip())
    )


def save_results():
    ordered_records = [
        saved_results[item["question_id"]]
        for item in questions
        if item["question_id"] in saved_results
    ]

    pd.DataFrame(
        ordered_records,
        columns=RESULT_COLUMNS
    ).to_csv(RAW_RESULTS_PATH, index=False)


client = genai.Client(api_key=API_KEY)

try:
    for item in questions:
        question_id = item["question_id"]

        previous = saved_results.get(question_id)

        if previous and is_complete(previous):
            print(f"{question_id}: Complete — reusing saved result")
            continue

        prompt = PROMPT_V1.replace(
            "{{question}}",
            item["question"]
        )

        print(f"\nProcessing {question_id}...")

        start_time = datetime.now(timezone.utc)
        start = time.perf_counter()

        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=0,
                    max_output_tokens=MAX_OUTPUT_TOKENS
                )
            )

            latency = time.perf_counter() - start
            end_time = datetime.now(timezone.utc)

            response_text = response.text or ""

            candidate = (
                response.candidates[0]
                if response.candidates else None
            )

            reason = (
                candidate.finish_reason
                if candidate is not None else None
            )

            finish_reason = (
                getattr(reason, "name", str(reason))
                if reason is not None else "UNKNOWN"
            )

            usage = response.usage_metadata

            if finish_reason == "STOP" and response_text.strip():
                status = "success"
            elif finish_reason == "MAX_TOKENS":
                status = "truncated"
            else:
                status = "incomplete"

            record = {
                "question_id": question_id,
                "question": item["question"],
                "prompt_version": "v1",
                "model_name": MODEL_NAME,
                "start_time_utc": start_time.isoformat(),
                "end_time_utc": end_time.isoformat(),
                "latency_seconds": round(latency, 4),
                "response_status": status,
                "finish_reason": finish_reason,
                "input_tokens": (
                    usage.prompt_token_count if usage else None
                ),
                "output_tokens": (
                    usage.candidates_token_count if usage else None
                ),
                "thinking_tokens": (
                    (usage.thoughts_token_count or 0)
                    if usage else None
                ),
                "total_tokens": (
                    usage.total_token_count if usage else None
                ),
                "response_text": response_text,
                "error_type": "",
                "max_output_tokens": MAX_OUTPUT_TOKENS
            }

            # Replace the complete record rather than modifying
            # individual DataFrame columns.
            saved_results[question_id] = record

            # Persist each measurement immediately.
            save_results()

            print("Status:", status)
            print("Finish reason:", finish_reason)
            print(f"Latency: {latency:.3f} seconds")
            print("Total tokens:", record["total_tokens"])

            if status != "success":
                print("Response needs review. Stopping execution.")
                break

        except Exception as exc:
            print(
                f"{question_id} failed: "
                f"{type(exc).__name__}: {str(exc)[:250]}"
            )
            print("Previously saved results remain preserved.")
            break

finally:
    client.close()


# Load finalized results for evaluation.
latency_df = pd.read_csv(RAW_RESULTS_PATH)

print("\n========== TASK 1 — FINAL RESULTS ==========")

display(
    latency_df[
        [
            "question_id",
            "prompt_version",
            "model_name",
            "latency_seconds",
            "response_status",
            "finish_reason",
            "input_tokens",
            "output_tokens",
            "thinking_tokens",
            "total_tokens",
            "max_output_tokens"
        ]
    ]
)

print("\n========== VERIFICATION ==========")

complete_count = sum(
    is_complete(record)
    for record in saved_results.values()
    if record.get("question_id") in {
        item["question_id"] for item in questions
    }
)

print(f"Completed responses: {complete_count}/5")
print("Saved to:", RAW_RESULTS_PATH)

if complete_count == 5:
    print("TASK 1 API EXECUTION: PASSED")
else:
    print("TASK 1 API EXECUTION: NEEDS REVIEW")



Processing Q1...
Status: success
Finish reason: STOP
Latency: 2.696 seconds
Total tokens: 577
Q2: Complete — reusing saved result

Processing Q3...
Status: success
Finish reason: STOP
Latency: 3.175 seconds
Total tokens: 649

Processing Q4...
Status: success
Finish reason: STOP
Latency: 2.681 seconds
Total tokens: 683

Processing Q5...
Status: success
Finish reason: STOP
Latency: 2.314 seconds
Total tokens: 503

========== TASK 1 — FINAL RESULTS ==========


,question_id,prompt_version,model_name,latency_seconds,response_status,finish_reason,input_tokens,output_tokens,thinking_tokens,total_tokens,max_output_tokens
0,Q1,v1,gemini-3.5-flash-lite,2.6960,success,STOP,21,556,0,577,2048
1,Q2,v1,gemini-3.5-flash-lite,1.3032,success,STOP,23,182,0,205,350
2,Q3,v1,gemini-3.5-flash-lite,3.1746,success,STOP,26,623,0,649,2048
3,Q4,v1,gemini-3.5-flash-lite,2.6815,success,STOP,22,661,0,683,2048
4,Q5,v1,gemini-3.5-flash-lite,2.3135,success,STOP,27,476,0,503,2048



========== VERIFICATION ==========
Completed responses: 5/5
Saved to: /Volumes/MLOpsWorkspace/eduassist-llmops-assignment2/outputs/task1_v1_raw.csv
TASK 1 API EXECUTION: PASSED


In [8]:

# Task 1 — Response Validation and Latency Statistics

task1_df = pd.read_csv(RAW_RESULTS_PATH)

print("========== RESPONSE VALIDATION ==========")

validation_df = task1_df[
    [
        "question_id",
        "response_status",
        "finish_reason",
        "latency_seconds"
    ]
].copy()

display(validation_df)

# Verify all five responses completed successfully.
assert len(task1_df) == 5
assert task1_df["question_id"].nunique() == 5
assert (task1_df["response_status"] == "success").all()
assert (task1_df["finish_reason"] == "STOP").all()

print("All five responses completed successfully.")

# Calculate the required latency statistics.
latencies = task1_df["latency_seconds"]

latency_summary = pd.DataFrame({
    "Metric": [
        "Mean",
        "Median",
        "Minimum",
        "Maximum",
        "Standard Deviation"
    ],
    "Latency (seconds)": [
        latencies.mean(),
        latencies.median(),
        latencies.min(),
        latencies.max(),
        latencies.std(ddof=1)
    ]
})

latency_summary["Latency (seconds)"] = (
    latency_summary["Latency (seconds)"].round(4)
)

print("\n========== LATENCY SUMMARY ==========")
display(latency_summary)

# Identify the slowest request.
slowest = task1_df.loc[
    task1_df["latency_seconds"].idxmax()
]

print("\n========== SLOWEST REQUEST ==========")
print("Question ID:", slowest["question_id"])
print("Question:", slowest["question"])
print("Latency:", slowest["latency_seconds"], "seconds")
print("Output tokens:", slowest["output_tokens"])

# Save summary for reproducibility.
LATENCY_SUMMARY_PATH = OUTPUT_DIR / "task1_latency_summary.csv"

latency_summary.to_csv(
    LATENCY_SUMMARY_PATH,
    index=False
)

print("\n========== FINAL VERIFICATION ==========")
print("Successful requests: 5/5")
print("Truncated responses: 0")
print("Latency summary saved:", LATENCY_SUMMARY_PATH)
print("TASK 1 VALIDATION: PASSED")


========== RESPONSE VALIDATION ==========


,question_id,response_status,finish_reason,latency_seconds
0,Q1,success,STOP,2.6960
1,Q2,success,STOP,1.3032
2,Q3,success,STOP,3.1746
3,Q4,success,STOP,2.6815
4,Q5,success,STOP,2.3135


All five responses completed successfully.

========== LATENCY SUMMARY ==========


,Metric,Latency (seconds)
0,Mean,2.4338
1,Median,2.6815
2,Minimum,1.3032
3,Maximum,3.1746
4,Standard Deviation,0.7021



========== SLOWEST REQUEST ==========
Question ID: Q3
Question: What is gradient descent, and why is it used to train neural networks?
Latency: 3.1746 seconds
Output tokens: 623

========== FINAL VERIFICATION ==========
Successful requests: 5/5
Truncated responses: 0
Latency summary saved: /Volumes/MLOpsWorkspace/eduassist-llmops-assignment2/outputs/task1_latency_summary.csv
TASK 1 VALIDATION: PASSED



### Task 1 — Latency Analysis and Interpretation

All five academic questions were successfully processed using the Gemini 3.5 Flash-Lite model with Prompt v1 (Basic). Each response completed normally with a `STOP` finish reason, confirming that none of the final responses were truncated.

The average response latency was **2.4338 seconds**, with a median of **2.6815 seconds**. The minimum and maximum latencies were **1.3032 seconds** and **3.1746 seconds**, respectively. The sample standard deviation was **0.7021 seconds**, indicating some variation in response times across the five requests.

Question Q3, which asked about gradient descent and its role in neural network training, was the slowest request at **3.1746 seconds**, generating 623 output tokens. Its relatively long response may have contributed to the higher latency. However, Q4 generated more output tokens while completing faster, suggesting that response length alone does not determine latency. Network conditions and variations in server-side processing are other possible factors.

Overall, the API successfully handled all five requests. Since these measurements are based on a small sample, additional requests would be needed to assess latency consistency more reliably under production conditions.



## Task 2: Count Tokens and Calculate Cost

This task analyzes token usage from the five successful Gemini API responses collected in Task 1.

The selected model is `gemini-3.5-flash-lite`. Its published standard paid-tier pricing, verified on November 2, 2026, is:

- Input tokens: **$0.30 per million tokens**
- Output tokens (including thinking tokens): **$2.50 per million tokens**
- Currency: **USD**

The project uses the Gemini API Free Tier. Therefore, actual free-tier token charges are estimated as zero for eligible requests. The paid-tier calculations are included to demonstrate API cost economics.

### Cost formulas

Input Cost = (Input Tokens / 1,000,000) × Input Price

Output Cost = (Billable Output Tokens / 1,000,000) × Output Price

Total Request Cost = Input Cost + Output Cost

Projected Monthly Cost = Average Request Cost × 10,000

Billable output tokens include generated output and any separately reported thinking tokens.

The projection assumes that future requests have similar token usage and are billed at standard paid-tier rates. It excludes additional services, taxes, and discounts.

**Pricing reference:** https://ai.google.dev/gemini-api/docs/pricing


In [9]:

# Task 2 — Count Tokens and Calculate Cost

import pandas as pd

INPUT_PRICE_PER_MILLION = 0.30
OUTPUT_PRICE_PER_MILLION = 2.50
MONTHLY_REQUESTS = 10000

PRICING_DATE = "2026-11-02"
CURRENCY = "USD"

# Reuse actual measurements collected in Task 1.
cost_df = pd.read_csv(RAW_RESULTS_PATH)

# Validate the source measurements.
assert len(cost_df) == 5
assert cost_df["question_id"].nunique() == 5
assert (cost_df["response_status"] == "success").all()
assert (cost_df["finish_reason"] == "STOP").all()
assert (cost_df["model_name"] == MODEL_NAME).all()

token_columns = [
    "input_tokens",
    "output_tokens",
    "thinking_tokens",
    "total_tokens"
]

for column in token_columns:
    cost_df[column] = pd.to_numeric(
        cost_df[column],
        errors="raise"
    )

assert cost_df[token_columns].notna().all().all()
assert (cost_df[token_columns] >= 0).all().all()

# Thinking tokens are included in output billing.
cost_df["billable_output_tokens"] = (
    cost_df["output_tokens"]
    + cost_df["thinking_tokens"]
)

# Calculate standard paid-tier costs in USD.
cost_df["input_cost_usd"] = (
    cost_df["input_tokens"]
    / 1_000_000
    * INPUT_PRICE_PER_MILLION
)

cost_df["output_cost_usd"] = (
    cost_df["billable_output_tokens"]
    / 1_000_000
    * OUTPUT_PRICE_PER_MILLION
)

cost_df["cost_per_request_usd"] = (
    cost_df["input_cost_usd"]
    + cost_df["output_cost_usd"]
)

# Estimated charges under eligible free-tier usage.
cost_df["free_tier_estimated_cost_usd"] = 0.0

# Record pricing information for reproducibility.
cost_df["input_price_per_million_usd"] = INPUT_PRICE_PER_MILLION
cost_df["output_price_per_million_usd"] = OUTPUT_PRICE_PER_MILLION
cost_df["pricing_date"] = PRICING_DATE
cost_df["currency"] = CURRENCY

print("========== TOKEN USAGE AND COST PER REQUEST ==========")

display_columns = [
    "question_id",
    "input_tokens",
    "output_tokens",
    "thinking_tokens",
    "total_tokens",
    "input_cost_usd",
    "output_cost_usd",
    "cost_per_request_usd"
]

display(cost_df[display_columns].round(7))

# Calculate aggregated measurements.
total_input_tokens = int(cost_df["input_tokens"].sum())
total_output_tokens = int(cost_df["output_tokens"].sum())
total_tokens = int(cost_df["total_tokens"].sum())

total_cost = cost_df["cost_per_request_usd"].sum()
average_cost = cost_df["cost_per_request_usd"].mean()

projected_monthly_cost = (
    average_cost * MONTHLY_REQUESTS
)

print("\n========== TOKEN SUMMARY ==========")
print("Total input tokens:", total_input_tokens)
print("Total output tokens:", total_output_tokens)
print("Total tokens:", total_tokens)

print("\n========== API COST SUMMARY ==========")
print(f"Total paid-tier estimated cost: ${total_cost:.7f}")
print(f"Average paid-tier cost per request: ${average_cost:.7f}")
print(f"Projected monthly requests: {MONTHLY_REQUESTS:,}")
print(f"Projected paid-tier monthly cost: ${projected_monthly_cost:.4f}")
print("Estimated eligible free-tier cost for observed requests: $0.00")
print("Currency:", CURRENCY)
print("Pricing date:", PRICING_DATE)

# Save the CSV required by the assignment.
COST_CSV_PATH = OUTPUT_DIR / "latency_tokens_cost.csv"

export_columns = [
    "question_id",
    "prompt_version",
    "model_name",
    "start_time_utc",
    "end_time_utc",
    "latency_seconds",
    "response_status",
    "finish_reason",
    "input_tokens",
    "output_tokens",
    "thinking_tokens",
    "billable_output_tokens",
    "total_tokens",
    "input_cost_usd",
    "output_cost_usd",
    "cost_per_request_usd",
    "free_tier_estimated_cost_usd",
    "input_price_per_million_usd",
    "output_price_per_million_usd",
    "pricing_date",
    "currency"
]

cost_df[export_columns].to_csv(
    COST_CSV_PATH,
    index=False
)

print("\n========== FINAL VERIFICATION ==========")

assert COST_CSV_PATH.exists()
assert (cost_df["cost_per_request_usd"] >= 0).all()

print("CSV file:", COST_CSV_PATH)
print("CSV rows:", len(cost_df))
print("API calls made: 0")
print("TASK 2 COST CALCULATION: PASSED")


========== TOKEN USAGE AND COST PER REQUEST ==========


,question_id,input_tokens,output_tokens,thinking_tokens,total_tokens,input_cost_usd,output_cost_usd,cost_per_request_usd
0,Q1,21,556,0,577,0.000006,0.001390,0.001396
1,Q2,23,182,0,205,0.000007,0.000455,0.000462
2,Q3,26,623,0,649,0.000008,0.001558,0.001565
3,Q4,22,661,0,683,0.000007,0.001653,0.001659
4,Q5,27,476,0,503,0.000008,0.001190,0.001198



========== TOKEN SUMMARY ==========
Total input tokens: 119
Total output tokens: 2498
Total tokens: 2617

========== API COST SUMMARY ==========
Total paid-tier estimated cost: $0.0062807
Average paid-tier cost per request: $0.0012561
Projected monthly requests: 10,000
Projected paid-tier monthly cost: $12.5614
Estimated eligible free-tier cost for observed requests: $0.00
Currency: USD
Pricing date: 2026-11-02

========== FINAL VERIFICATION ==========
CSV file: /Volumes/MLOpsWorkspace/eduassist-llmops-assignment2/outputs/latency_tokens_cost.csv
CSV rows: 5
API calls made: 0
TASK 2 COST CALCULATION: PASSED



### Task 2 — Cost Analysis and Optimization Recommendation

The five academic questions consumed a total of **2,617 tokens**, consisting of 119 input tokens and 2,498 output tokens. No thinking tokens were reported.

Using the published standard paid-tier pricing for Gemini 3.5 Flash-Lite, the estimated total cost of the five requests was **\$0.0062807**, with an average cost of **\$0.00125614 per request**. At an estimated volume of 10,000 similar requests per month, the projected paid-tier cost would be **\$12.56**.

The project currently uses the Gemini API Free Tier, so the estimated charge for eligible requests is \$0. The paid-tier projection is included to demonstrate how token-based billing works.

**Cost Optimization Recommendation:**

Output tokens accounted for approximately **99.43% of the estimated paid-tier cost**. Therefore, the most useful optimization would be to reduce unnecessary response length.

For example, the prompt could instruct the model to provide concise academic explanations with a limited number of key points. This would help reduce generated tokens and potentially improve latency.

If output token consumption were reduced by 30%, while input usage remained unchanged, the projected monthly paid-tier cost would decrease from approximately **\$12.56 to \$8.81**, a saving of about **\$3.75 per month**.

However, shorter responses should still be evaluated for clarity, completeness, and accuracy. Reducing token usage should not compromise the quality of answers provided to students.



## Task 3 — Prompt Versioning and MLflow Evaluation

The EduAssist AI application requires a reliable prompt for answering undergraduate academic questions.

Three prompt versions are registered in the MLflow Prompt Registry:

- **v1 — Basic:** Provides clear answers without additional instructions.
- **v2 — Structured:** Adds an academic assistant role and instructions for concise, undergraduate-level explanations.
- **v3 — JSON Output:** Requires answers containing `answer`, `confidence`, and `key_points` fields in JSON format.

All three prompts use the same `{{question}}` placeholder and will be evaluated using the same five questions.

Prompt versioning allows changes to be tracked, compared, and reproduced. The best-performing version will be assigned the `production` alias after evaluation.

The evaluation considers JSON validity, response completeness, latency, token usage, and estimated API cost.


In [10]:

# Task 3 — Register Three Prompt Versions in MLflow

import mlflow
import pandas as pd

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_registry_uri(TRACKING_URI)

PROMPT_NAME = "eduassist_academic_qa"

# Prompt v1 — Basic
PROMPT_V1 = """Answer the following student question clearly.
Question: {{question}}"""

# Prompt v2 — Structured
PROMPT_V2 = """You are an academic assistant.
Answer the following student question in a clear, concise, and accurate manner.
Use simple language suitable for undergraduate students.
Question: {{question}}"""

# Prompt v3 — JSON Output
PROMPT_V3 = """You are an academic assistant.
Answer the student question and return ONLY valid JSON.
Do not include markdown, explanations outside JSON, or code fences.

Use this exact JSON schema:
{
  "answer": "string",
  "confidence": "high | medium | low",
  "key_points": ["string"]
}
Question: {{question}}"""

prompt_definitions = [
    {
        "version": 1,
        "label": "Basic",
        "template": PROMPT_V1,
        "commit_message": "Initial basic academic question-answering prompt"
    },
    {
        "version": 2,
        "label": "Structured",
        "template": PROMPT_V2,
        "commit_message": "Add academic role and concise undergraduate-level instructions"
    },
    {
        "version": 3,
        "label": "JSON Output",
        "template": PROMPT_V3,
        "commit_message": "Require JSON answer, confidence and key_points fields"
    }
]

registered_prompts = []

print("========== MLFLOW PROMPT REGISTRATION ==========")
print("Prompt name:", PROMPT_NAME)
print("Registry URI:", mlflow.get_registry_uri())

for definition in prompt_definitions:

    version = definition["version"]

    prompt_uri = f"prompts:/{PROMPT_NAME}/{version}"

    # Check whether this exact version already exists.
    existing_prompt = mlflow.genai.load_prompt(
        prompt_uri,
        allow_missing=True,
        link_to_model=False
    )

    if existing_prompt is None:

        prompt = mlflow.genai.register_prompt(
            name=PROMPT_NAME,
            template=definition["template"],
            commit_message=definition["commit_message"],
            tags={
                "project": "EduAssist AI",
                "assignment": "MLOps Assignment 2",
                "prompt_type": definition["label"]
            }
        )

        assert int(prompt.version) == version, (
            f"Expected version {version}, "
            f"but MLflow created version {prompt.version}"
        )

        action = "CREATED"

    else:
        if existing_prompt.template != definition["template"]:
            raise ValueError(
                f"Existing prompt version {version} differs "
                "from the assignment template."
            )

        prompt = existing_prompt
        action = "ALREADY EXISTS"

    # Verify that the prompt can be loaded from MLflow.
    loaded_prompt = mlflow.genai.load_prompt(
        prompt_uri,
        link_to_model=False
    )

    assert loaded_prompt.template == definition["template"]

    # Verify the question placeholder is working.
    rendered_prompt = loaded_prompt.format(
        question=questions[0]["question"]
    )

    assert questions[0]["question"] in rendered_prompt
    assert "{{question}}" not in rendered_prompt

    registered_prompts.append({
        "version": f"v{version}",
        "name": loaded_prompt.name,
        "type": definition["label"],
        "registry_version": loaded_prompt.version,
        "status": action,
        "commit_message": definition["commit_message"]
    })

    print(f"\nPrompt v{version} — {definition['label']}")
    print("Status:", action)
    print("Registry version:", loaded_prompt.version)
    print("Template verification: PASSED")
    print("Placeholder rendering: PASSED")

print("\n========== REGISTERED PROMPT VERSIONS ==========")

registry_df = pd.DataFrame(registered_prompts)

display(registry_df)

print("\n========== FINAL VERIFICATION ==========")

assert len(registry_df) == 3
assert registry_df["registry_version"].astype(int).tolist() == [1, 2, 3]

print("Registered versions: 3")
print("Prompt version sequence: v1, v2, v3")
print("All templates verified.")
print("Gemini API requests made: 0")
print("PROMPT REGISTRATION: PASSED")


========== MLFLOW PROMPT REGISTRATION ==========
Prompt name: eduassist_academic_qa
Registry URI: http://127.0.0.1:5050


2026/11/02 02:22:17 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: eduassist_academic_qa, version 1
2026/11/02 02:22:17 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: eduassist_academic_qa, version 2
2026/11/02 02:22:17 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: eduassist_academic_qa, version 3



Prompt v1 — Basic
Status: CREATED
Registry version: 1
Template verification: PASSED
Placeholder rendering: PASSED

Prompt v2 — Structured
Status: CREATED
Registry version: 2
Template verification: PASSED
Placeholder rendering: PASSED

Prompt v3 — JSON Output
Status: CREATED
Registry version: 3
Template verification: PASSED
Placeholder rendering: PASSED

========== REGISTERED PROMPT VERSIONS ==========


,version,name,type,registry_version,status,commit_message
0,v1,eduassist_academic_qa,Basic,1,CREATED,Initial basic academic question-answering prompt
1,v2,eduassist_academic_qa,Structured,2,CREATED,Add academic role and concise undergraduate-le...
2,v3,eduassist_academic_qa,JSON Output,3,CREATED,"Require JSON answer, confidence and key_points..."



========== FINAL VERIFICATION ==========
Registered versions: 3
Prompt version sequence: v1, v2, v3
All templates verified.
Gemini API requests made: 0
PROMPT REGISTRATION: PASSED



### Task 3 — Prompt v2 Evaluation

Prompt v2 introduces an academic assistant role and requests clear, concise, and accurate explanations suitable for undergraduate students.

The five questions from Task 1 are reused to ensure a consistent comparison.

For each request, we record response latency, input tokens, output tokens, total tokens, response status, and generated text.

A response is considered complete only when it contains nonempty text and has a `STOP` finish reason.

The results are saved to CSV for subsequent comparison with Prompt v1 and Prompt v3.


In [12]:

# Task 3 — Evaluate Prompt v2 (Structured)

import time
from datetime import datetime, timezone

V2_RESULTS_PATH = OUTPUT_DIR / "task3_v2_raw.csv"
TASK3_MAX_OUTPUT_TOKENS = 2048

# Load the actual registered prompt from MLflow.
prompt_v2 = mlflow.genai.load_prompt(
    f"prompts:/{PROMPT_NAME}/2",
    link_to_model=False
)

assert int(prompt_v2.version) == 2

# Reuse completed responses if the cell is rerun.
saved_v2_results = {}

if V2_RESULTS_PATH.exists():
    existing = pd.read_csv(
        V2_RESULTS_PATH,
        keep_default_na=False
    )

    saved_v2_results = {
        str(row["question_id"]): row
        for row in existing.to_dict("records")
    }

print("========== PROMPT V2 EVALUATION ==========")
print("Prompt:", PROMPT_NAME)
print("Registry version:", prompt_v2.version)
print("Model:", MODEL_NAME)
print("Questions:", len(questions))
print("Maximum output tokens:", TASK3_MAX_OUTPUT_TOKENS)

client = genai.Client(api_key=API_KEY)

try:
    for item in questions:

        question_id = item["question_id"]
        previous = saved_v2_results.get(question_id)

        if (
            previous
            and previous.get("response_status") == "success"
            and previous.get("finish_reason") == "STOP"
            and str(previous.get("response_text", "")).strip()
        ):
            print(f"\n{question_id}: Complete — reusing saved result")
            continue

        # Render the registered MLflow prompt.
        prompt_text = prompt_v2.format(
            question=item["question"]
        )

        print(f"\nProcessing {question_id}...")

        start_time = datetime.now(timezone.utc)
        start = time.perf_counter()

        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt_text,
                config=types.GenerateContentConfig(
                    temperature=0,
                    max_output_tokens=TASK3_MAX_OUTPUT_TOKENS
                )
            )

            latency = time.perf_counter() - start
            end_time = datetime.now(timezone.utc)

            response_text = response.text or ""

            candidate = (
                response.candidates[0]
                if response.candidates else None
            )

            reason = (
                candidate.finish_reason
                if candidate is not None else None
            )

            finish_reason = (
                getattr(reason, "name", str(reason))
                if reason is not None else "UNKNOWN"
            )

            usage = response.usage_metadata

            if finish_reason == "STOP" and response_text.strip():
                status = "success"
            elif finish_reason == "MAX_TOKENS":
                status = "truncated"
            else:
                status = "incomplete"

            record = {
                "question_id": question_id,
                "question": item["question"],
                "prompt_version": "v2",
                "model_name": MODEL_NAME,
                "start_time_utc": start_time.isoformat(),
                "end_time_utc": end_time.isoformat(),
                "latency_seconds": round(latency, 4),
                "response_status": status,
                "finish_reason": finish_reason,
                "input_tokens": (
                    usage.prompt_token_count if usage else None
                ),
                "output_tokens": (
                    usage.candidates_token_count if usage else None
                ),
                "thinking_tokens": (
                    (usage.thoughts_token_count or 0)
                    if usage else None
                ),
                "total_tokens": (
                    usage.total_token_count if usage else None
                ),
                "response_text": response_text,
                "error_type": "",
                "max_output_tokens": TASK3_MAX_OUTPUT_TOKENS
            }

            saved_v2_results[question_id] = record

            # Preserve progress after every request.
            ordered_results = [
                saved_v2_results[q["question_id"]]
                for q in questions
                if q["question_id"] in saved_v2_results
            ]

            pd.DataFrame(ordered_results).to_csv(
                V2_RESULTS_PATH,
                index=False
            )

            print("Status:", status)
            print("Finish reason:", finish_reason)
            print(f"Latency: {latency:.3f} seconds")
            print("Total tokens:", record["total_tokens"])

            if status != "success":
                print("Incomplete response detected. Stopping evaluation.")
                break

        except Exception as exc:
            print(
                f"Request failed: {type(exc).__name__}: "
                f"{str(exc)[:200]}"
            )
            print("Stopping to avoid unnecessary API requests.")
            break

finally:
    client.close()

print("\n========== PROMPT V2 RESULTS ==========")

v2_df = pd.read_csv(V2_RESULTS_PATH)

display(
    v2_df[
        [
            "question_id",
            "prompt_version",
            "latency_seconds",
            "response_status",
            "finish_reason",
            "input_tokens",
            "output_tokens",
            "thinking_tokens",
            "total_tokens"
        ]
    ]
)

print("\n========== VALIDATION ==========")

complete_count = (
    (v2_df["response_status"] == "success")
    & (v2_df["finish_reason"] == "STOP")
).sum()

print(f"Completed responses: {complete_count}/5")
print("Results saved to:", V2_RESULTS_PATH)

if complete_count == 5:
    print("PROMPT V2 EVALUATION: PASSED")
else:
    print("PROMPT V2 EVALUATION: NEEDS REVIEW")


========== PROMPT V2 EVALUATION ==========
Prompt: eduassist_academic_qa
Registry version: 2
Model: gemini-3.5-flash-lite
Questions: 5
Maximum output tokens: 2048

Processing Q1...
Status: success
Finish reason: STOP
Latency: 1.826 seconds
Total tokens: 309

Processing Q2...
Status: success
Finish reason: STOP
Latency: 1.171 seconds
Total tokens: 217

Processing Q3...
Status: success
Finish reason: STOP
Latency: 1.863 seconds
Total tokens: 398

Processing Q4...
Status: success
Finish reason: STOP
Latency: 1.699 seconds
Total tokens: 385

Processing Q5...
Status: success
Finish reason: STOP
Latency: 2.349 seconds
Total tokens: 390

========== PROMPT V2 RESULTS ==========


,question_id,prompt_version,latency_seconds,response_status,finish_reason,input_tokens,output_tokens,thinking_tokens,total_tokens
0,Q1,v2,1.8258,success,STOP,45,264,0,309
1,Q2,v2,1.1713,success,STOP,47,170,0,217
2,Q3,v2,1.8629,success,STOP,50,348,0,398
3,Q4,v2,1.6990,success,STOP,46,339,0,385
4,Q5,v2,2.3488,success,STOP,51,339,0,390



========== VALIDATION ==========
Completed responses: 5/5
Results saved to: /Volumes/MLOpsWorkspace/eduassist-llmops-assignment2/outputs/task3_v2_raw.csv
PROMPT V2 EVALUATION: PASSED



### Task 3 — Prompt v3 Evaluation and JSON Validation

Prompt v3 instructs Gemini to return only JSON containing the fields `answer`, `confidence`, and `key_points`.

The same five academic questions are evaluated using the registered MLflow Prompt v3.

Two validation checks are performed:

- **valid_json:** Whether the response can be parsed using Python's `json.loads()`.
- **schema_valid:** Whether the response contains exactly the required keys, a nonempty string answer, an allowed confidence value (`high`, `medium`, or `low`), and a list of strings for key points.

The JSON validity rate is calculated as:

**Valid JSON Rate = Number of Valid JSON Responses / Total Responses**

JSON syntax validity and schema validity are measured separately. A response may contain valid JSON without satisfying the required output structure.

The experiment uses temperature 0 and a maximum of 2048 output tokens, matching the Prompt v2 evaluation settings.


In [13]:

# Task 3 — Evaluate Prompt v3 with JSON Validation

import json
import time
from datetime import datetime, timezone

V3_RESULTS_PATH = OUTPUT_DIR / "task3_v3_raw.csv"
TASK3_MAX_OUTPUT_TOKENS = 2048

# Load Prompt v3 from the MLflow Prompt Registry.
prompt_v3 = mlflow.genai.load_prompt(
    f"prompts:/{PROMPT_NAME}/3",
    link_to_model=False
)

assert int(prompt_v3.version) == 3


def validate_json_response(response_text):
    """Return JSON validity, schema validity, and error details."""

    try:
        parsed = json.loads(response_text)
    except (json.JSONDecodeError, TypeError) as exc:
        return False, False, str(exc)

    expected_keys = {"answer", "confidence", "key_points"}

    schema_valid = (
        isinstance(parsed, dict)
        and set(parsed.keys()) == expected_keys
        and isinstance(parsed["answer"], str)
        and bool(parsed["answer"].strip())
        and isinstance(parsed["confidence"], str)
        and parsed["confidence"] in {"high", "medium", "low"}
        and isinstance(parsed["key_points"], list)
        and all(
            isinstance(point, str)
            for point in parsed["key_points"]
        )
    )

    return True, schema_valid, ""


# Reuse previously completed responses, even if their JSON is invalid.
# This prevents cherry-picking responses to inflate the pass rate.
saved_v3_results = {}

if V3_RESULTS_PATH.exists():
    existing = pd.read_csv(
        V3_RESULTS_PATH,
        keep_default_na=False
    )
    saved_v3_results = {
        str(row["question_id"]): row
        for row in existing.to_dict("records")
    }

print("========== PROMPT V3 JSON EVALUATION ==========")
print("Prompt:", PROMPT_NAME)
print("Registry version:", prompt_v3.version)
print("Model:", MODEL_NAME)
print("Questions:", len(questions))
print("Maximum output tokens:", TASK3_MAX_OUTPUT_TOKENS)

client = genai.Client(api_key=API_KEY)

try:
    for item in questions:

        question_id = item["question_id"]
        previous = saved_v3_results.get(question_id)

        if (
            previous
            and previous.get("response_status") == "success"
            and previous.get("finish_reason") == "STOP"
            and str(previous.get("response_text", "")).strip()
        ):
            print(f"\n{question_id}: Reusing completed response")
            continue

        prompt_text = prompt_v3.format(
            question=item["question"]
        )

        print(f"\nProcessing {question_id}...")

        start_time = datetime.now(timezone.utc)
        start = time.perf_counter()

        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt_text,
                config=types.GenerateContentConfig(
                    temperature=0,
                    max_output_tokens=TASK3_MAX_OUTPUT_TOKENS
                )
            )

            latency = time.perf_counter() - start
            end_time = datetime.now(timezone.utc)

            response_text = response.text or ""

            candidate = (
                response.candidates[0]
                if response.candidates else None
            )

            reason = (
                candidate.finish_reason
                if candidate is not None else None
            )

            finish_reason = (
                getattr(reason, "name", str(reason))
                if reason is not None else "UNKNOWN"
            )

            usage = response.usage_metadata

            if finish_reason == "STOP" and response_text.strip():
                status = "success"
            elif finish_reason == "MAX_TOKENS":
                status = "truncated"
            else:
                status = "incomplete"

            valid_json, schema_valid, json_error = (
                validate_json_response(response_text)
            )

            record = {
                "question_id": question_id,
                "question": item["question"],
                "prompt_version": "v3",
                "model_name": MODEL_NAME,
                "start_time_utc": start_time.isoformat(),
                "end_time_utc": end_time.isoformat(),
                "latency_seconds": round(latency, 4),
                "response_status": status,
                "finish_reason": finish_reason,
                "input_tokens": (
                    usage.prompt_token_count if usage else None
                ),
                "output_tokens": (
                    usage.candidates_token_count if usage else None
                ),
                "thinking_tokens": (
                    (usage.thoughts_token_count or 0)
                    if usage else None
                ),
                "total_tokens": (
                    usage.total_token_count if usage else None
                ),
                "response_text": response_text,
                "error_type": "",
                "max_output_tokens": TASK3_MAX_OUTPUT_TOKENS,
                "valid_json": valid_json,
                "schema_valid": schema_valid,
                "json_error": json_error
            }

            saved_v3_results[question_id] = record

            # Save progress after each API request.
            ordered_results = [
                saved_v3_results[q["question_id"]]
                for q in questions
                if q["question_id"] in saved_v3_results
            ]

            pd.DataFrame(ordered_results).to_csv(
                V3_RESULTS_PATH,
                index=False
            )

            print("Status:", status)
            print("Finish reason:", finish_reason)
            print(f"Latency: {latency:.3f} seconds")
            print("Total tokens:", record["total_tokens"])
            print("Valid JSON:", valid_json)
            print("Schema valid:", schema_valid)

            if status != "success":
                print("Incomplete response. Stopping for review.")
                break

        except Exception as exc:
            print(
                f"Request failed: {type(exc).__name__}: "
                f"{str(exc)[:200]}"
            )
            print("Stopping to avoid unnecessary requests.")
            break

finally:
    client.close()


print("\n========== PROMPT V3 RESULTS ==========")

v3_df = pd.read_csv(V3_RESULTS_PATH)

display(
    v3_df[
        [
            "question_id",
            "latency_seconds",
            "response_status",
            "finish_reason",
            "input_tokens",
            "output_tokens",
            "total_tokens",
            "valid_json",
            "schema_valid"
        ]
    ]
)

print("\n========== JSON VALIDATION SUMMARY ==========")

total_responses = len(v3_df)
valid_count = int(v3_df["valid_json"].sum())
schema_count = int(v3_df["schema_valid"].sum())

print("Total responses:", total_responses)
print("Valid JSON responses:", valid_count)
print("Schema-valid responses:", schema_count)

print(f"JSON validity rate: {valid_count / total_responses:.2%}")
print(f"Schema validity rate: {schema_count / total_responses:.2%}")

complete_count = int(
    (
        (v3_df["response_status"] == "success")
        & (v3_df["finish_reason"] == "STOP")
    ).sum()
)

print("\n========== FINAL VERIFICATION ==========")
print(f"Completed responses: {complete_count}/5")
print("Results saved to:", V3_RESULTS_PATH)

if total_responses == 5 and complete_count == 5:
    print("PROMPT V3 EVALUATION: PASSED")
else:
    print("PROMPT V3 EVALUATION: NEEDS REVIEW")


========== PROMPT V3 JSON EVALUATION ==========
Prompt: eduassist_academic_qa
Registry version: 3
Model: gemini-3.5-flash-lite
Questions: 5
Maximum output tokens: 2048

Processing Q1...
Status: success
Finish reason: STOP
Latency: 1.585 seconds
Total tokens: 214
Valid JSON: True
Schema valid: True

Processing Q2...
Status: success
Finish reason: STOP
Latency: 1.112 seconds
Total tokens: 226
Valid JSON: True
Schema valid: True

Processing Q3...
Status: success
Finish reason: STOP
Latency: 1.251 seconds
Total tokens: 228
Valid JSON: True
Schema valid: True

Processing Q4...
Status: success
Finish reason: STOP
Latency: 1.265 seconds
Total tokens: 261
Valid JSON: True
Schema valid: True

Processing Q5...
Status: success
Finish reason: STOP
Latency: 1.178 seconds
Total tokens: 228
Valid JSON: True
Schema valid: True

========== PROMPT V3 RESULTS ==========


,question_id,latency_seconds,response_status,finish_reason,input_tokens,output_tokens,total_tokens,valid_json,schema_valid
0,Q1,1.5850,success,STOP,86,128,214,True,True
1,Q2,1.1122,success,STOP,88,138,226,True,True
2,Q3,1.2511,success,STOP,91,137,228,True,True
3,Q4,1.2655,success,STOP,87,174,261,True,True
4,Q5,1.1778,success,STOP,92,136,228,True,True



========== JSON VALIDATION SUMMARY ==========
Total responses: 5
Valid JSON responses: 5
Schema-valid responses: 5
JSON validity rate: 100.00%
Schema validity rate: 100.00%

========== FINAL VERIFICATION ==========
Completed responses: 5/5
Results saved to: /Volumes/MLOpsWorkspace/eduassist-llmops-assignment2/outputs/task3_v3_raw.csv
PROMPT V3 EVALUATION: PASSED



### Task 3 — Prompt Comparison and Evaluation Metrics

The three registered prompt versions are compared using their saved responses to the same five academic questions.

The required evaluation metrics are:

- `avg_latency_seconds`: Average API response time.
- `avg_input_tokens`: Average prompt tokens per request.
- `avg_output_tokens`: Average generated output tokens per request.
- `avg_cost_per_request`: Average estimated paid-tier API cost in USD.
- `valid_json`: Proportion of responses that contain syntactically valid JSON.

An additional metric, `schema_valid_rate`, measures compliance with the required EduAssist JSON structure.

Cost calculations use the same published model pricing as Task 2. No additional API requests are necessary because the original response measurements are reused.

JSON validity is calculated for all three prompt versions, not only v3, to allow a consistent comparison.


In [14]:

# Task 3 — Compare All Three Prompt Versions

import pandas as pd

RESULT_FILES = {
    "v1": OUTPUT_DIR / "task1_v1_raw.csv",
    "v2": OUTPUT_DIR / "task3_v2_raw.csv",
    "v3": OUTPUT_DIR / "task3_v3_raw.csv"
}

all_results = []
expected_ids = {q["question_id"] for q in questions}

for version, path in RESULT_FILES.items():

    assert path.exists(), f"Missing results: {path}"

    df = pd.read_csv(path, keep_default_na=False)

    assert len(df) == 5, f"{version}: Expected 5 responses"
    assert set(df["question_id"]) == expected_ids
    assert df["question_id"].is_unique
    assert (df["response_status"] == "success").all()
    assert (df["finish_reason"] == "STOP").all()
    assert (df["model_name"] == MODEL_NAME).all()

    df["prompt_version"] = version
    df["prompt_registry_uri"] = (
        f"prompts:/{PROMPT_NAME}/{version[1:]}"
    )

    # Validate JSON consistently for every prompt version.
    validations = [
        validate_json_response(str(text))
        for text in df["response_text"]
    ]

    df["valid_json"] = [v[0] for v in validations]
    df["schema_valid"] = [v[1] for v in validations]

    # Convert token and latency measurements to numeric values.
    numeric_columns = [
        "latency_seconds",
        "input_tokens",
        "output_tokens",
        "thinking_tokens",
        "total_tokens"
    ]

    for column in numeric_columns:
        df[column] = pd.to_numeric(
            df[column],
            errors="raise"
        )

    assert df[numeric_columns].notna().all().all()
    assert (df[numeric_columns] >= 0).all().all()

    # Same pricing method used in Task 2.
    df["billable_output_tokens"] = (
        df["output_tokens"] + df["thinking_tokens"]
    )

    df["input_cost_usd"] = (
        df["input_tokens"]
        * INPUT_PRICE_PER_MILLION / 1_000_000
    )

    df["output_cost_usd"] = (
        df["billable_output_tokens"]
        * OUTPUT_PRICE_PER_MILLION / 1_000_000
    )

    df["cost_per_request_usd"] = (
        df["input_cost_usd"] + df["output_cost_usd"]
    )

    df["pricing_date"] = PRICING_DATE
    df["currency"] = CURRENCY

    all_results.append(df)

# Combine all 15 responses.
evaluation_df = pd.concat(
    all_results,
    ignore_index=True
)

assert len(evaluation_df) == 15

# Calculate aggregate evaluation metrics.
comparison_df = (
    evaluation_df
    .groupby("prompt_version", sort=True)
    .agg(
        total_requests=("question_id", "count"),
        avg_latency_seconds=("latency_seconds", "mean"),
        avg_input_tokens=("input_tokens", "mean"),
        avg_output_tokens=("output_tokens", "mean"),
        avg_total_tokens=("total_tokens", "mean"),
        avg_cost_per_request=("cost_per_request_usd", "mean"),
        valid_json=("valid_json", "mean"),
        schema_valid_rate=("schema_valid", "mean")
    )
    .reset_index()
)

# Add model and registry details.
comparison_df["model_name"] = MODEL_NAME

comparison_df["prompt_registry_uri"] = (
    comparison_df["prompt_version"].apply(
        lambda v: f"prompts:/{PROMPT_NAME}/{v[1:]}"
    )
)

# Save the assignment-required evaluation CSV.
EVALUATION_CSV_PATH = (
    OUTPUT_DIR / "prompt_evaluation_results.csv"
)

COMPARISON_CSV_PATH = (
    OUTPUT_DIR / "prompt_comparison.csv"
)

evaluation_df.to_csv(
    EVALUATION_CSV_PATH,
    index=False
)

comparison_df.to_csv(
    COMPARISON_CSV_PATH,
    index=False
)

print("========== PROMPT COMPARISON ==========")

display(
    comparison_df[
        [
            "prompt_version",
            "avg_latency_seconds",
            "avg_input_tokens",
            "avg_output_tokens",
            "avg_total_tokens",
            "avg_cost_per_request",
            "valid_json",
            "schema_valid_rate"
        ]
    ].round(6)
)

print("\n========== JSON VALIDITY BY VERSION ==========")

for _, row in comparison_df.iterrows():
    print(
        f"{row['prompt_version']}: "
        f"Valid JSON = {row['valid_json']:.0%}, "
        f"Schema valid = {row['schema_valid_rate']:.0%}"
    )

print("\n========== OUTPUT FILES ==========")
print("Evaluation CSV:", EVALUATION_CSV_PATH)
print("Comparison CSV:", COMPARISON_CSV_PATH)

print("\n========== FINAL VALIDATION ==========")

assert len(comparison_df) == 3
assert (comparison_df["total_requests"] == 5).all()
assert evaluation_df.groupby("prompt_version").size().eq(5).all()
assert EVALUATION_CSV_PATH.exists()
assert COMPARISON_CSV_PATH.exists()

print("Total evaluated responses:", len(evaluation_df))
print("Prompt versions:", len(comparison_df))
print("API requests made: 0")
print("PROMPT COMPARISON: PASSED")


========== PROMPT COMPARISON ==========


,prompt_version,avg_latency_seconds,avg_input_tokens,avg_output_tokens,avg_total_tokens,avg_cost_per_request,valid_json,schema_valid_rate
0,v1,2.43376,23.8,499.6,523.4,0.001256,0.0,0.0
1,v2,1.78156,47.8,292.0,339.8,0.000744,0.0,0.0
2,v3,1.27832,88.8,142.6,231.4,0.000383,1.0,1.0



========== JSON VALIDITY BY VERSION ==========
v1: Valid JSON = 0%, Schema valid = 0%
v2: Valid JSON = 0%, Schema valid = 0%
v3: Valid JSON = 100%, Schema valid = 100%

========== OUTPUT FILES ==========
Evaluation CSV: /Volumes/MLOpsWorkspace/eduassist-llmops-assignment2/outputs/prompt_evaluation_results.csv
Comparison CSV: /Volumes/MLOpsWorkspace/eduassist-llmops-assignment2/outputs/prompt_comparison.csv

========== FINAL VALIDATION ==========
Total evaluated responses: 15
Prompt versions: 3
API requests made: 0
PROMPT COMPARISON: PASSED



### Task 3 — Logging Prompt Evaluation Metrics in MLflow

Each prompt version is represented by a separate MLflow experiment run.

The metrics are calculated from the saved evaluations rather than making additional API requests.

The required metrics are:

- `avg_latency_seconds`
- `avg_input_tokens`
- `avg_output_tokens`
- `avg_cost_per_request`
- `valid_json`

Additional metrics include average total tokens, schema validity, and response completion rate.

Each run also records the Gemini model, prompt version, generation settings, and pricing details. The original response CSV and combined evaluation results are logged as artifacts.

Previously recorded runs are reused when the notebook is executed again to avoid duplicate experiment entries.


In [15]:

# Task 3 — Log Prompt Evaluation Metrics in MLflow

import mlflow
import pandas as pd
from mlflow.tracking import MlflowClient

EXPERIMENT_NAME = "eduassist_llmops_assignment"

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_registry_uri(TRACKING_URI)

experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

assert experiment is not None, "MLflow experiment not found"
assert len(comparison_df) == 3
assert len(evaluation_df) == 15

tracking_client = MlflowClient(tracking_uri=TRACKING_URI)

# Find previously logged assignment evaluation runs.
existing_runs = tracking_client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.evaluation_stage = 'prompt_comparison'",
    max_results=100
)

run_records = []

print("========== MLFLOW EXPERIMENT LOGGING ==========")
print("Experiment:", EXPERIMENT_NAME)
print("Experiment ID:", experiment.experiment_id)
print("Model:", MODEL_NAME)

for version in ["v1", "v2", "v3"]:

    row = comparison_df.loc[
        comparison_df["prompt_version"] == version
    ].iloc[0]

    prompt_uri = (
        f"prompts:/{PROMPT_NAME}/{version[1:]}"
    )

    # Reuse an existing evaluation run when available.
    matching_runs = [
        run for run in existing_runs
        if run.data.tags.get("prompt_version") == version
    ]

    if matching_runs:
        run_context = mlflow.start_run(
            run_id=matching_runs[0].info.run_id
        )
        action = "UPDATED"
    else:
        run_context = mlflow.start_run(
            experiment_id=experiment.experiment_id,
            run_name=f"prompt_{version}_evaluation"
        )
        action = "CREATED"

    with run_context as run:

        # Load the registered prompt inside the active run.
        registered_prompt = mlflow.genai.load_prompt(
            prompt_uri,
            link_to_model=False
        )

        assert int(registered_prompt.version) == int(version[1:])

        # Record experiment metadata.
        mlflow.set_tags({
            "evaluation_stage": "prompt_comparison",
            "prompt_version": version,
            "prompt_registry_uri": prompt_uri,
            "project": "EduAssist AI",
            "assignment": "MLOps Assignment 2",
            "cost_basis": "estimated_standard_paid_tier",
            "currency": CURRENCY,
            "pricing_date": PRICING_DATE
        })

        # Record reproducibility parameters.
        mlflow.log_params({
            "model_name": MODEL_NAME,
            "prompt_name": PROMPT_NAME,
            "prompt_version": version,
            "temperature": 0,
            "question_count": 5,
            "input_price_per_million": INPUT_PRICE_PER_MILLION,
            "output_price_per_million": OUTPUT_PRICE_PER_MILLION
        })

        # The original v1 measurements used mixed token limits.
        generation_limit = (
            "350_Q2_2048_others"
            if version == "v1"
            else "2048"
        )

        mlflow.set_tag(
            "max_output_tokens_configuration",
            generation_limit
        )

        metrics = {
            "avg_latency_seconds": float(
                row["avg_latency_seconds"]
            ),
            "avg_input_tokens": float(
                row["avg_input_tokens"]
            ),
            "avg_output_tokens": float(
                row["avg_output_tokens"]
            ),
            "avg_cost_per_request": float(
                row["avg_cost_per_request"]
            ),
            "valid_json": float(
                row["valid_json"]
            ),
            "avg_total_tokens": float(
                row["avg_total_tokens"]
            ),
            "schema_valid_rate": float(
                row["schema_valid_rate"]
            ),
            "response_completion_rate": 1.0
        }

        mlflow.log_metrics(metrics)

        # Log original measurements for this prompt version.
        mlflow.log_artifact(
            str(RESULT_FILES[version]),
            artifact_path="responses"
        )

        # Log the consolidated evaluation evidence.
        mlflow.log_artifact(
            str(EVALUATION_CSV_PATH),
            artifact_path="evaluation"
        )

        mlflow.log_artifact(
            str(COMPARISON_CSV_PATH),
            artifact_path="evaluation"
        )

        run_id = run.info.run_id

    # Verify the metrics were persisted to MLflow.
    saved_run = tracking_client.get_run(run_id)

    for metric_name, expected_value in metrics.items():
        assert metric_name in saved_run.data.metrics

        actual_value = saved_run.data.metrics[metric_name]

        assert abs(actual_value - expected_value) < 1e-9

    run_records.append({
        "prompt_version": version,
        "run_id": run_id,
        "action": action,
        "avg_latency_seconds": metrics["avg_latency_seconds"],
        "avg_cost_per_request": metrics["avg_cost_per_request"],
        "valid_json": metrics["valid_json"],
        "schema_valid_rate": metrics["schema_valid_rate"]
    })

    print(f"\nPrompt {version}")
    print("Run status:", action)
    print("Metrics logged:", len(metrics))
    print("Registry prompt:", prompt_uri)
    print("Run verification: PASSED")


print("\n========== MLFLOW RUN SUMMARY ==========")

mlflow_runs_df = pd.DataFrame(run_records)

display(
    mlflow_runs_df.drop(columns=["run_id"]).round(6)
)

print("\n========== FINAL VALIDATION ==========")

assert len(mlflow_runs_df) == 3
assert mlflow_runs_df["run_id"].nunique() == 3

print("Experiment runs:", len(mlflow_runs_df))
print("Unique run IDs:", mlflow_runs_df["run_id"].nunique())
print("Metrics per run: 8")
print("Gemini API requests made: 0")
print("MLFLOW METRIC LOGGING: PASSED")


========== MLFLOW EXPERIMENT LOGGING ==========
Experiment: eduassist_llmops_assignment
Experiment ID: 1
Model: gemini-3.5-flash-lite
🏃 View run prompt_v1_evaluation at: http://127.0.0.1:5050/#/experiments/1/runs/f8b10d39ac2d4ae0baf7558667cf91c3
🧪 View experiment at: http://127.0.0.1:5050/#/experiments/1

Prompt v1
Run status: CREATED
Metrics logged: 8
Registry prompt: prompts:/eduassist_academic_qa/1
Run verification: PASSED
🏃 View run prompt_v2_evaluation at: http://127.0.0.1:5050/#/experiments/1/runs/1a9d86ca9bf044138890ed48ec1b0137
🧪 View experiment at: http://127.0.0.1:5050/#/experiments/1

Prompt v2
Run status: CREATED
Metrics logged: 8
Registry prompt: prompts:/eduassist_academic_qa/2
Run verification: PASSED
🏃 View run prompt_v3_evaluation at: http://127.0.0.1:5050/#/experiments/1/runs/7cf0f9659a3f46cabd4ef554b9103c0f
🧪 View experiment at: http://127.0.0.1:5050/#/experiments/1

Prompt v3
Run status: CREATED
Metrics logged: 8
Registry prompt: prompts:/eduassist_academic_qa/3
Run

,prompt_version,action,avg_latency_seconds,avg_cost_per_request,valid_json,schema_valid_rate
0,v1,CREATED,2.43376,0.001256,0.0,0.0
1,v2,CREATED,1.78156,0.000744,0.0,0.0
2,v3,CREATED,1.27832,0.000383,1.0,1.0



========== FINAL VALIDATION ==========
Experiment runs: 3
Unique run IDs: 3
Metrics per run: 8
Gemini API requests made: 0
MLFLOW METRIC LOGGING: PASSED



### Task 3 — Production Prompt Promotion

The three prompt versions were compared using JSON validity, schema compliance, response latency, token consumption, and estimated API cost.

Prompt v3 was selected for production because it achieved:

- 100% JSON validity (5 out of 5 responses).
- 100% compliance with the required JSON schema.
- The lowest average latency at 1.27832 seconds.
- The lowest average output-token usage at 142.6 tokens.
- The lowest average estimated paid-tier cost at $0.00038314 per request.

The `production` alias is assigned to Prompt Registry version 3.

The alias is verified by loading the prompt directly from MLflow and checking that it resolves to version 3.

This promotion represents the selected configuration for the assignment. A real production deployment would also require broader factual accuracy, safety, and reliability evaluation.


In [16]:

# Task 3 — Promote Best Prompt to Production

import mlflow

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_registry_uri(TRACKING_URI)

PRODUCTION_VERSION = 3
PRODUCTION_ALIAS = "production"

print("========== PRODUCTION PROMOTION ==========")

# Step 1: Validate the evaluation results.
assert len(comparison_df) == 3
assert set(comparison_df["prompt_version"]) == {"v1", "v2", "v3"}

best = comparison_df.loc[
    comparison_df["prompt_version"] == "v3"
].iloc[0]

assert best["valid_json"] == 1.0
assert best["schema_valid_rate"] == 1.0

assert (
    best["avg_latency_seconds"]
    == comparison_df["avg_latency_seconds"].min()
)

assert (
    best["avg_cost_per_request"]
    == comparison_df["avg_cost_per_request"].min()
)

print("Selected prompt: v3 — JSON Output")
print("JSON validity:", f"{best['valid_json']:.0%}")
print("Schema validity:", f"{best['schema_valid_rate']:.0%}")
print(
    "Average latency:",
    f"{best['avg_latency_seconds']:.5f} seconds"
)
print(
    "Average estimated cost:",
    f"${best['avg_cost_per_request']:.8f}"
)
print("Evaluation checks: PASSED")

# Step 2: Assign the production alias.
mlflow.genai.set_prompt_alias(
    name=PROMPT_NAME,
    alias=PRODUCTION_ALIAS,
    version=PRODUCTION_VERSION
)

print("\nProduction alias assigned.")

# Step 3: Load the prompt using the production alias.
production_uri = (
    f"prompts:/{PROMPT_NAME}@{PRODUCTION_ALIAS}"
)

production_prompt = mlflow.genai.load_prompt(
    production_uri,
    link_to_model=False,
    cache_ttl_seconds=0
)

# Step 4: Compare the aliased prompt with registered v3.
registered_v3 = mlflow.genai.load_prompt(
    f"prompts:/{PROMPT_NAME}/3",
    link_to_model=False
)

assert int(production_prompt.version) == 3
assert production_prompt.name == PROMPT_NAME
assert production_prompt.template == registered_v3.template

# Step 5: Verify that the production prompt renders correctly.
test_question = questions[0]["question"]

rendered_prompt = production_prompt.format(
    question=test_question
)

assert test_question in rendered_prompt
assert "{{question}}" not in rendered_prompt

print("\n========== PRODUCTION ALIAS VERIFICATION ==========")
print("Prompt name:", production_prompt.name)
print("Production alias:", PRODUCTION_ALIAS)
print("Resolved version:", production_prompt.version)
print("Production URI:", production_uri)
print("Template comparison: PASSED")
print("Question rendering: PASSED")

print("\n========== FINAL VERIFICATION ==========")
print("Promoted version: v3")
print("Gemini API requests made: 0")
print("PRODUCTION PROMOTION: PASSED")


========== PRODUCTION PROMOTION ==========
Selected prompt: v3 — JSON Output
JSON validity: 100%
Schema validity: 100%
Average latency: 1.27832 seconds
Average estimated cost: $0.00038314
Evaluation checks: PASSED

Production alias assigned.

========== PRODUCTION ALIAS VERIFICATION ==========
Prompt name: eduassist_academic_qa
Production alias: production
Resolved version: 3
Production URI: prompts:/eduassist_academic_qa@production
Template comparison: PASSED
Question rendering: PASSED

========== FINAL VERIFICATION ==========
Promoted version: v3
Gemini API requests made: 0
PRODUCTION PROMOTION: PASSED



### Task 3 — Prompt Comparison and Production Promotion Justification

#### Final Prompt Comparison

| Metric | v1 — Basic | v2 — Structured | v3 — JSON Output |
|---|---:|---:|---:|
| Average Latency (seconds) | 2.4338 | 1.7816 | 1.2783 |
| Average Input Tokens | 23.8 | 47.8 | 88.8 |
| Average Output Tokens | 499.6 | 292.0 | 142.6 |
| Average Total Tokens | 523.4 | 339.8 | 231.4 |
| Average Estimated Cost (USD) | 0.00125614 | 0.00074434 | 0.00038314 |
| Valid JSON Rate | 0% | 0% | 100% |
| Schema Validity Rate | 0% | 0% | 100% |
| Recommended? | No | No | **Yes** |

#### Production Promotion Justification

After evaluating all three prompt versions using the same five academic questions, **Prompt v3 (JSON Output)** was selected for production because it best satisfied the EduAssist AI application's requirements while also demonstrating better operational efficiency.

**Response Quality and JSON Validity:** Prompt v1 generated general explanations, while v2 introduced clearer instructions for concise academic responses. However, neither version produced valid JSON in the five evaluated requests. Prompt v3 achieved **100% JSON validity and 100% schema compliance**, successfully returning the required `answer`, `confidence`, and `key_points` fields. This makes its responses more suitable for integration with applications that expect structured data. Nevertheless, JSON compliance does not establish factual accuracy, which was not independently evaluated.

**Latency:** Prompt v3 had the lowest average response latency at **1.27832 seconds**, compared with 1.78156 seconds for v2 and 2.43376 seconds for v1. This represents approximately a 47.5% reduction compared with v1.

**Token Consumption:** Although v3 required more input tokens because of its detailed JSON instructions, it generated substantially fewer output tokens. Its average total token usage was **231.4 tokens**, compared with 339.8 for v2 and 523.4 for v1.

**Cost Efficiency:** Prompt v3 also had the lowest estimated paid-tier cost at **$0.00038314 per request**, approximately 69.5% lower than v1. These are estimated costs based on published pricing; the evaluation used an eligible Gemini Free Tier project.

Considering its JSON compliance, lower latency, reduced token consumption, and lower estimated cost, **Prompt v3 was promoted by assigning the `production` alias in the MLflow Prompt Registry**.

The evaluation was limited to five questions per version.
